# VORTEX KI ベクトル化ノートブック

**ベクトルモデル**: `Qwen/Qwen3-Embedding-8B` (4096d)

KI アーティファクト (`~/.gemini/antigravity/knowledge/`) から
`ki_vectors_8bit.db` を生成する。

## フロー
1. VORTEX サイドバー → 「KI Colab Notebook を開く」
2. Run All
3. 結果: `ki_vectors_8bit.db` → Memory Recall の埋め込みバックエンド

## 動作環境
- **Colab**: int8 量子化 (bitsandbytes) — GPU メモリ節約
- **ローカル**: fp16 — Mac Studio M3 Ultra / 512GB RAM


In [ ]:
# 依存インストール
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import os
import json
import glob
import sqlite3
import time
from pathlib import Path

# ─── Colab / ローカル検出 ─────────────────────────────────────────────
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except Exception as exc:
    print(f'Colab ランタイム未検出: {exc}')

# ─── Qwen3-Embedding-8B 設定 ──────────────────────────────────────────
MODEL_NAME = 'Qwen/Qwen3-Embedding-8B'
DIMENSIONS = 4096
QUERY_INSTRUCTION = (
    'Retrieve relevant technical documentation about '
    'AI cognitive architecture, security systems, and development infrastructure'
)
MAX_CHARS = 8000
BATCH_SIZE = 4

# ─── パス解決 ─────────────────────────────────────────────────────────
KI_BASE_CANDIDATES = [
    '/content/drive/MyDrive/.gemini/antigravity/knowledge',
    '/content/drive/MyDrive/antigravity/knowledge',
    os.path.expanduser('~/.gemini/antigravity/knowledge'),
]
KI_BASE = next(
    (p for p in KI_BASE_CANDIDATES if os.path.isdir(p)),
    KI_BASE_CANDIDATES[-1]
)

WORKDIR = '/content/newgate_ki' if IN_COLAB else str(Path.cwd() / 'newgate_ki')
DB_OUTPUT = os.path.join(WORKDIR, 'ki_vectors_8bit.db')
DRIVE_OUTPUT = '/content/drive/MyDrive/Newgate/ki_vectors_8bit.db'
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'IN_COLAB': IN_COLAB,
    'MODEL_NAME': MODEL_NAME,
    'DIMENSIONS': DIMENSIONS,
    'KI_BASE': KI_BASE,
    'DB_OUTPUT': DB_OUTPUT,
}, indent=2))

In [ ]:
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import torch
import torch.nn.functional as F

def log(msg: str) -> None:
    print(f'[{time.strftime("%H:%M:%S")}] {msg}')


def collect_ki_documents(ki_base: str):
    """knowledge/ 配下の metadata.json + artifacts/*.md を収集"""
    documents = []
    for ki_dir in sorted(glob.glob(os.path.join(ki_base, '*'))):
        if not os.path.isdir(ki_dir):
            continue
        meta_path = os.path.join(ki_dir, 'metadata.json')
        artifacts_dir = os.path.join(ki_dir, 'artifacts')
        if not os.path.exists(meta_path) or not os.path.isdir(artifacts_dir):
            continue
        with open(meta_path, 'r', encoding='utf-8') as fh:
            meta = json.load(fh)
        ki_name = os.path.basename(ki_dir)
        for root, _, files in os.walk(artifacts_dir):
            for fname in sorted(files):
                if not fname.endswith('.md'):
                    continue
                fpath = os.path.join(root, fname)
                with open(fpath, 'r', encoding='utf-8') as fh:
                    content = fh.read()
                if len(content.strip()) < 50:
                    continue
                documents.append({
                    'ki_name': ki_name,
                    'ki_title': meta.get('title', ki_name),
                    'artifact_path': os.path.relpath(fpath, ki_dir),
                    'content_length': len(content),
                    'full_text': content,
                })
    return documents


def chunk_text(text: str, max_chars: int = MAX_CHARS):
    """段落境界でチャンク分割"""
    if len(text) <= max_chars:
        return [text]
    chunks, current = [], ''
    for para in text.split('\n\n'):
        if len(current) + len(para) > max_chars and current:
            chunks.append(current.strip())
            current = para
        else:
            current = current + '\n\n' + para if current else para
    if current.strip():
        chunks.append(current.strip())
    return chunks or [text[:max_chars]]


def open_database(db_path: str):
    conn = sqlite3.connect(db_path)
    conn.execute('DROP TABLE IF EXISTS ki_chunks')
    conn.execute('DROP TABLE IF EXISTS metadata')
    conn.execute('''
        CREATE TABLE ki_chunks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ki_name TEXT NOT NULL,
            ki_title TEXT NOT NULL,
            artifact_path TEXT NOT NULL,
            chunk_index INTEGER NOT NULL,
            chunk_length INTEGER NOT NULL,
            content_preview TEXT NOT NULL,
            content_text TEXT NOT NULL,
            vector_json TEXT NOT NULL,
            dims INTEGER NOT NULL,
            model TEXT NOT NULL,
            precision TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    ''')
    conn.execute('CREATE INDEX idx_ki_name ON ki_chunks(ki_name)')
    conn.execute('CREATE INDEX idx_artifact ON ki_chunks(artifact_path)')
    conn.execute('CREATE TABLE metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
    return conn


def load_model():
    """Qwen3-Embedding-8B を int8 (Colab) または fp16 (ローカル) でロード"""
    kwargs = {
        'trust_remote_code': True,
        'device_map': 'auto',
        'torch_dtype': torch.float16,
    }
    if IN_COLAB:
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
        log('Colab: int8 量子化で Qwen3-Embedding-8B をロード')
    else:
        log('ローカル: fp16 で Qwen3-Embedding-8B をロード')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModel.from_pretrained(MODEL_NAME, **kwargs)
    model.eval()
    return tokenizer, model


def _model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def encode_texts(model, tokenizer, texts):
    """テキストをベクトル化 (4096d)"""
    if hasattr(model, 'encode'):
        embeddings = model.encode(
            texts,
            instruction=QUERY_INSTRUCTION,
            max_length=MAX_CHARS,
        )
        if isinstance(embeddings, torch.Tensor):
            embeddings = embeddings.detach().cpu()
        return [list(map(float, row)) for row in embeddings]

    prompt_texts = [
        f'Instruct: {QUERY_INSTRUCTION}\nQuery: {t[:MAX_CHARS]}'
        for t in texts
    ]
    device = _model_device(model)
    batch = tokenizer(
        prompt_texts, padding=True, truncation=True,
        max_length=8192, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model(**batch)
    last_hidden = out.last_hidden_state
    mask = batch['attention_mask'].unsqueeze(-1).float()
    pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    normed = F.normalize(pooled, p=2, dim=1)
    return [list(map(float, row)) for row in normed.cpu()]

In [ ]:
def run_vectorization():
    log(f'KI ベース: {KI_BASE}')
    documents = collect_ki_documents(KI_BASE)
    log(f'KI ドキュメント数: {len(documents)}')
    if not documents:
        log('⚠ ドキュメントがありません。KI_BASE を確認してください。')
        return

    tokenizer, model = load_model()
    log('モデルロード完了')

    all_chunks = []
    for doc in documents:
        chunks = chunk_text(doc['full_text'])
        for ci, chunk in enumerate(chunks):
            all_chunks.append({
                'ki_name': doc['ki_name'],
                'ki_title': doc['ki_title'],
                'artifact_path': doc['artifact_path'],
                'chunk_index': ci,
                'text': chunk,
            })
    log(f'チャンク数: {len(all_chunks)}')

    conn = open_database(DB_OUTPUT)
    now_iso = time.strftime('%Y-%m-%dT%H:%M:%S')
    precision = 'int8' if IN_COLAB else 'fp16'
    total, t0 = 0, time.time()

    for i in range(0, len(all_chunks), BATCH_SIZE):
        batch = all_chunks[i:i + BATCH_SIZE]
        texts = [c['text'] for c in batch]
        vectors = encode_texts(model, tokenizer, texts)
        for c, vec in zip(batch, vectors):
            conn.execute(
                '''INSERT INTO ki_chunks
                   (ki_name, ki_title, artifact_path, chunk_index, chunk_length,
                    content_preview, content_text, vector_json, dims, model, precision, created_at)
                   VALUES (?,?,?,?,?,?,?,?,?,?,?,?)''',
                (
                    c['ki_name'], c['ki_title'], c['artifact_path'],
                    c['chunk_index'], len(c['text']),
                    c['text'][:200], c['text'],
                    json.dumps(vec), len(vec),
                    MODEL_NAME, precision, now_iso,
                )
            )
            total += 1
        if (i // BATCH_SIZE) % 10 == 0:
            log(f'  進捗: {total}/{len(all_chunks)} チャンク')
        conn.commit()

    elapsed = time.time() - t0
    conn.execute(
        "INSERT INTO metadata VALUES ('run_info', ?)",
        (json.dumps({
            'model': MODEL_NAME,
            'dimensions': DIMENSIONS,
            'precision': precision,
            'total_chunks': total,
            'total_documents': len(documents),
            'elapsed_seconds': round(elapsed, 1),
            'ki_base': KI_BASE,
            'created_at': now_iso,
        }, ensure_ascii=False),)
    )
    conn.commit()
    conn.close()

    db_size_mb = os.path.getsize(DB_OUTPUT) / (1024 * 1024)
    log(f'完了: {total} チャンク, {db_size_mb:.1f} MB, {elapsed:.1f} 秒')
    log(f'出力: {DB_OUTPUT}')

run_vectorization()

In [ ]:
# Colab: Drive にコピー
if IN_COLAB and os.path.exists(DB_OUTPUT):
    import shutil
    Path(DRIVE_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DB_OUTPUT, DRIVE_OUTPUT)
    log(f'Drive にコピー完了: {DRIVE_OUTPUT}')
    log(f'サイズ: {os.path.getsize(DRIVE_OUTPUT) / (1024*1024):.1f} MB')
else:
    log(f'ローカル出力: {DB_OUTPUT}')
    log('Mac Studio ではこのまま memory_pipeline recall に接続可能')